# Chapter 5 &mdash; Why the Blow-Up is Unavoidable

**Concept 11 of the Chapter 5 decomposition:** *Why the Blow-Up is Unavoidable: the Look-Back Language $L_{Nthlast1}$*

$\{x1y : y\in\{0,1\}^{N-1}\}$ needs exponentially many states &mdash; distinguishable suffixes force them.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter5/Concept-Blow-Up-Unavoidable/Concept-Blow-Up-Unavoidable.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


The blow-up is not a failure of imagination. For
$L_{Nthlast1} = \{x1y : x\in\{0,1\}^*,\ y\in\{0,1\}^{N-1}\}$
**every** DFA needs at least $2^{N}$ states.

The argument is a **distinguishability** argument (formalised in Chapter 6 as
Myhill&ndash;Nerode): if two distinct bit-patterns $u \ne v$ of length $N$ led to the same
state, pick a position where they differ and feed a suffix that makes one accept and
the other reject. Contradiction. So all $2^N$ patterns need distinct states.

This is the same lower-bound style used throughout complexity theory.

## 2. Definitions

### The distinguishing suffix, constructed

In [ ]:
def distinguisher(u, v):
    """For u != v of the same length N, a suffix z with exactly one of uz, vz in L."""
    N = len(u)
    i = next(k for k in range(N) if u[k] != v[k])
    return '0' * i            # pushes position i to be the N-th from the end

### Membership test for the N-th-last-is-1 language

In [ ]:
def in_LN(s, N): return len(s) >= N and s[-N] == '1'

## 3. Tests

Any two distinct length-$N$ prefixes are **distinguishable**.

In [ ]:
from itertools import product
N = 4
pats = [''.join(p) for p in product('01', repeat=N)]
count = 0
for u, v in ((a, b) for i, a in enumerate(pats) for b in pats[i+1:]):
    z = distinguisher(u, v)
    assert in_LN(u + z, N) != in_LN(v + z, N), (u, v, z)
    count += 1
print("all %d pairs of length-%d patterns separated by an explicit suffix" % (count, N))

Hence at least $2^N$ states, and Jove's minimizer confirms it.

In [ ]:
def nth_last(N):
    lines = ['DFA']
    wins = [''.join(p) for p in product('01', repeat=N)]
    def nm(w): return ('F_' if w[0] == '1' else 'S_') + w
    for k in range(N):
        for p in product('01', repeat=k):
            src = 'I' if k == 0 else 'S_' + ''.join(p)
            for b in '01':
                t = ''.join(p) + b
                lines.append('%s : %s -> %s' % (src, b, nm(t) if len(t) == N else 'S_' + t))
    for w in wins:
        for b in '01':
            lines.append('%s : %s -> %s' % (nm(w), b, nm((w + b)[1:])))
    return md2mc('\n'.join(lines))

for N in range(1, 5):
    m = len(min_dfa(nth_last(N))["Q"])
    print("N=%d : lower bound 2^N = %2d, minimal DFA has %2d states" % (N, 2**N, m))
    assert m >= 2**N

A worked separation, spelled out.

In [ ]:
u, v, N = '1000', '0000', 4
z = distinguisher(u, v)
print("u=%s v=%s  suffix z=%r" % (u, v, z))
print("  uz = %-8r in L? %s" % (u+z, in_LN(u+z, N)))
print("  vz = %-8r in L? %s" % (v+z, in_LN(v+z, N)))
print("\nOne accepts, one rejects -> u and v CANNOT share a state.")

## 4. Exercises


1. Give the distinguishing suffix for $u=0110$, $v=0010$ with $N=4$.
2. Why does the argument need $u$ and $v$ to have the *same* length?
3. Look ahead: how does Myhill&ndash;Nerode package this argument?

In [ ]:
# Your work for the exercises above.